In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [2]:
credentials_location = '/home/codespace/.config/gcloud/application_default_credentials.json'

gcs_jar = '/workspaces/etl-zoomcamp-project/lib/gcs-connector-hadoop3-2.2.5.jar'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('my-etl-project') \
    .set("spark.jars", gcs_jar) \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [3]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

26/03/22 18:00:35 WARN Utils: Your hostname, codespaces-b411cf resolves to a loopback address: 127.0.0.1; using 10.0.1.104 instead (on interface eth0)
26/03/22 18:00:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/22 18:00:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 18:00:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [5]:
df = spark.read.csv("gs://mental-health-etl-bucket/raw/unemployment_data.csv", header=True)

In [6]:
df.show()

26/03/22 18:00:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+------------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|        Country Name|Country Code| 1991| 1992| 1993| 1994| 1995| 1996| 1997| 1998| 1999| 2000| 2001| 2002| 2003| 2004| 2005| 2006| 2007| 2008| 2009| 2010| 2011| 2012| 2013| 2014| 2015| 2016| 2017| 2018| 2019| 2020| 2021|
+--------------------+------------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|Africa Eastern an...|         AFE|  7.8| 7.84| 7.85| 7.84| 7.83| 7.84| 7.86| 7.81| 7.79| 7.72| 7.73| 7.96| 7.79| 7.31| 7.12| 6.99| 6.74| 6.27| 6.32| 6.87| 6.75| 6.56| 6.45| 6.41| 6.49| 6.61| 6.71| 6.73| 6.91| 7.56| 8.11|
|         Afghanistan|         AFG|10.65|10.82|10.72|10.73|11.18|10.96|10.78| 10.8|10.81|10.81|10.81|11.26|11.14

In [7]:
df.printSchema()

root
 |-- Country Name: string (nullable = true)
 |-- Country Code: string (nullable = true)
 |-- 1991: string (nullable = true)
 |-- 1992: string (nullable = true)
 |-- 1993: string (nullable = true)
 |-- 1994: string (nullable = true)
 |-- 1995: string (nullable = true)
 |-- 1996: string (nullable = true)
 |-- 1997: string (nullable = true)
 |-- 1998: string (nullable = true)
 |-- 1999: string (nullable = true)
 |-- 2000: string (nullable = true)
 |-- 2001: string (nullable = true)
 |-- 2002: string (nullable = true)
 |-- 2003: string (nullable = true)
 |-- 2004: string (nullable = true)
 |-- 2005: string (nullable = true)
 |-- 2006: string (nullable = true)
 |-- 2007: string (nullable = true)
 |-- 2008: string (nullable = true)
 |-- 2009: string (nullable = true)
 |-- 2010: string (nullable = true)
 |-- 2011: string (nullable = true)
 |-- 2012: string (nullable = true)
 |-- 2013: string (nullable = true)
 |-- 2014: string (nullable = true)
 |-- 2015: string (nullable = true)
 |-- 20

In [18]:
from pyspark.sql import functions as F

# Lista godina
year_columns = [str(year) for year in range(1991, 2022)]

# Melt operacija i konverzija tipova
df_melted = df.select(
    F.col("Country Name").alias("country"),
    F.expr("stack({0}, {1}) as (year, unemployment_rate)".format(
        len(year_columns),
        ', '.join([f"'{col}', `{col}`" for col in year_columns])
    ))
).withColumn("year", F.col("year").cast("int")) \
 .withColumn("unemployment_rate", F.col("unemployment_rate").cast("double"))

df_melted.show(5)

+--------------------+----+-----------------+
|             country|year|unemployment_rate|
+--------------------+----+-----------------+
|Africa Eastern an...|1991|              7.8|
|Africa Eastern an...|1992|             7.84|
|Africa Eastern an...|1993|             7.85|
|Africa Eastern an...|1994|             7.84|
|Africa Eastern an...|1995|             7.83|
+--------------------+----+-----------------+
only showing top 5 rows



In [19]:
df_melted.printSchema()

root
 |-- country: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- unemployment_rate: double (nullable = true)



In [23]:
df_melted.count()

7285

In [24]:
from pyspark.sql.functions import when, col

df_clean = df_melted.withColumn(
    "country",
    when(col("country") == "Russian Federation", "Russia")
    .when(col("country") == "Eswatini", "Swaziland")
    .when(col("country") == "Turkiye", "Turkey")
    .when(col("country") == "Korea, Dem. People's Rep.", "North Korea")
    .when(col("country") == "Korea, Rep.", "South Korea")
    .when(col("country") == "Hong Kong SAR, China", "Hong Kong")
    .when(col("country") == "Macao SAR, China", "Macao")
    .when(col("country") == "Timor-Leste", "Timor")
    .when(col("country") == "North Macedonia", "Macedonia")
    .when(col("country") == "West Bank and Gaza", "Palestine")
    .when(col("country") == "Venezuela, RB", "Venezuela")
    .when(col("country") == "Iran, Islamic Rep.", "Iran")
    .when(col("country") == "Bahamas, The", "Bahamas")
    .when(col("country") == "Gambia, The", "Gambia")
    .when(col("country") == "Slovak Republic", "Slovakia")
    .when(col("country") == "St. Lucia", "Saint Lucia")
    .when(col("country") == "St. Vincent and the Grenadines", "Saint Vincent and the Grenadines")
    .otherwise(col("country"))
)

In [25]:
df_clean.filter(col("country") == "Slovakia").show()

+--------+----+-----------------+
| country|year|unemployment_rate|
+--------+----+-----------------+
|Slovakia|1991|            11.58|
|Slovakia|1992|            11.87|
|Slovakia|1993|             12.2|
|Slovakia|1994|            13.65|
|Slovakia|1995|            13.11|
|Slovakia|1996|            11.34|
|Slovakia|1997|            11.89|
|Slovakia|1998|            12.19|
|Slovakia|1999|            15.95|
|Slovakia|2000|            19.06|
|Slovakia|2001|            19.38|
|Slovakia|2002|            18.72|
|Slovakia|2003|            17.12|
|Slovakia|2004|             18.6|
|Slovakia|2005|            16.26|
|Slovakia|2006|            13.37|
|Slovakia|2007|            11.14|
|Slovakia|2008|             9.51|
|Slovakia|2009|            12.03|
|Slovakia|2010|            14.38|
+--------+----+-----------------+
only showing top 20 rows



In [26]:
df_clean.printSchema()

root
 |-- country: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- unemployment_rate: double (nullable = true)



In [27]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_clean.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_clean.columns])
null_counts.show()

+-------+----+-----------------+
|country|year|unemployment_rate|
+-------+----+-----------------+
|      0|   0|                0|
+-------+----+-----------------+



In [28]:
df_clean.write.mode("overwrite").parquet("gs://mental-health-etl-bucket/processed/unemployment_data.parquet")